In [1]:
!pip install requests gtfs-realtime-bindings


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import requests
from google.transit import gtfs_realtime_pb2

# TTC GTFS-Realtime Vehicle Positions feed
GTFS_RT_URL = "https://bustime.ttc.ca/gtfsrt/vehicles"

# Download the feed
response = requests.get(GTFS_RT_URL)
response.raise_for_status()

# Parse GTFS-RT protobuf
feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

# Find and display ONE bus with location data
for entity in feed.entity:
    if entity.HasField("vehicle"):
        vehicle = entity.vehicle

        if vehicle.position.latitude and vehicle.position.longitude:
            print("🚍 TTC Bus Live Position")
            print("-----------------------")
            print(f"Vehicle ID : {vehicle.vehicle.id}")
            print(f"Route ID   : {vehicle.trip.route_id}")
            print(f"Latitude   : {vehicle.position.latitude}")
            print(f"Longitude  : {vehicle.position.longitude}")
            print(f"Timestamp  : {vehicle.timestamp}")
            break

🚍 TTC Bus Live Position
-----------------------
Vehicle ID : 3638
Route ID   : 84
Latitude   : 43.75318908691406
Longitude  : -79.44957733154297
Timestamp  : 1765673497


In [5]:
import requests
from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict

GTFS_RT_URL = "https://bustime.ttc.ca/gtfsrt/vehicles"

response = requests.get(GTFS_RT_URL)
response.raise_for_status()

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

for entity in feed.entity:
    if entity.HasField("vehicle"):
        vehicle_dict = MessageToDict(
            entity.vehicle,
            preserving_proto_field_name=True
        )
        print(vehicle_dict)
        break

{'trip': {'trip_id': '57943070', 'schedule_relationship': 'SCHEDULED', 'route_id': '84'}, 'position': {'latitude': 43.755318, 'longitude': -79.43891, 'bearing': 80.0, 'speed': 0.44704}, 'timestamp': '1765673619', 'vehicle': {'id': '3638'}, 'occupancy_status': 'FEW_SEATS_AVAILABLE'}


In [1]:
def extract_vehicle_features(v):
    return {
        # Trip
        "trip_id": getattr(v.trip, "trip_id", None),
        "route_id": getattr(v.trip, "route_id", None),
        "direction_id": getattr(v.trip, "direction_id", None),
        "start_time": getattr(v.trip, "start_time", None),
        "start_date": getattr(v.trip, "start_date", None),
        "schedule_relationship": str(v.trip.schedule_relationship),

        # Vehicle
        "vehicle_id": getattr(v.vehicle, "id", None),
        "vehicle_label": getattr(v.vehicle, "label", None),

        # Position
        "latitude": getattr(v.position, "latitude", None),
        "longitude": getattr(v.position, "longitude", None),
        "bearing": getattr(v.position, "bearing", None),
        "speed_mps": getattr(v.position, "speed", None),

        # Status
        "current_stop_sequence": getattr(v, "current_stop_sequence", None),
        "stop_id": getattr(v, "stop_id", None),
        "current_status": str(v.current_status),

        # Extras
        "occupancy_status": str(v.occupancy_status),
        "timestamp": getattr(v, "timestamp", None),
    }

In [2]:
import time
import requests
import folium
from IPython.display import display, clear_output
from google.transit import gtfs_realtime_pb2

GTFS_RT_URL = "https://bustime.ttc.ca/gtfsrt/vehicles"
ROUTE_ID = "16"   # ← change route here

while True:
    response = requests.get(GTFS_RT_URL)
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    buses = []

    for entity in feed.entity:
        if entity.HasField("vehicle"):
            v = entity.vehicle
            if v.trip.route_id == ROUTE_ID and v.position.latitude:
                buses.append(v)

    if not buses:
        print(f"No buses found for Route {ROUTE_ID}")
        time.sleep(60)
        continue

    # Center map on first bus
    m = folium.Map(
        location=[buses[0].position.latitude, buses[0].position.longitude],
        zoom_start=14,
        tiles="CartoDB positron"
    )

    for v in buses:
        speed = round(v.position.speed, 2) if v.position.speed else "N/A"
        bearing = round(v.position.bearing, 1) if v.position.bearing else "N/A"

        popup = f"""
        <b>Route:</b> {v.trip.route_id}<br>
        <b>Vehicle ID:</b> {v.vehicle.id}<br>
        <b>Trip ID:</b> {v.trip.trip_id}<br>
        <b>Direction:</b> {v.trip.direction_id}<br>
        <b>Speed:</b> {speed} m/s<br>
        <b>Bearing:</b> {bearing}°
        """

        folium.Marker(
            location=[v.position.latitude, v.position.longitude],
            popup=popup,
            icon=folium.Icon(icon="bus", prefix="fa")
        ).add_to(m)

    clear_output(wait=True)
    display(m)

    time.sleep(60)


KeyboardInterrupt: 